# cAPTure: development OOF operational evaluation

This notebook compares XGB-P, both context ablations, and full XGB-P+T under the same predeclared false-alert budgets. It reads only checksum-verified development OOF artifacts. A separate score threshold is selected for each model and budget from development OOF scores; held-out author-train and final-test scenario contents are not read.


## 1. Prepare the Colab environment


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", "--branch", REPOSITORY_BRANCH,
                    "--single-branch", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
branch = subprocess.check_output(["git", "branch", "--show-current"],
                                 cwd=PROJECT_ROOT, text=True).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")
required_files = ["code/python/requirements-capture-xgb.txt",
                  "code/python/utils/capture_xgb_p_t.py",
                  "code/python/utils/capture_oof_operational.py",
                  "configs/capture_experiment_v1.yaml"]
missing = [name for name in required_files if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Update the Colab repository copy first: {missing}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(PROJECT_ROOT / "code/python/requirements-capture-xgb.txt")], check=True)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))
import pandas as pd
from IPython.display import display
print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip())


Mounted at /content/drive
Repository commit: 5f7a771e66d5d36eda0e3383965ca63d0d2ed8b4


## 2. Bind the completed model runs

Enter the completed XGB-P+T and ablation run IDs. Set `EVALUATION_RUN_ID` only when resuming an existing evaluation. The evaluator validates fold assignments, artifact checksums, packet counts, context provenance, and the frozen manifest policy before reporting metrics.


In [ ]:
from utils.capture_oof_operational import (
    run_operational_oof_evaluation, validate_operational_run,
)

MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
BASELINE_RUN_DIR = DRIVE_ROOT / "xgb_p_runs/20260919T151844_852400Z_xgb_p"
PRIMARY_RUN_ID = "20260919T201909_754628Z_xgb_p_t"
ABLATION_RUN_ID = "20260920T000745_916321Z_xgb_p_t_ablation"
EVALUATION_RUN_ID = None  # Set only when resuming an existing evaluation.
if PRIMARY_RUN_ID is None or ABLATION_RUN_ID is None:
    raise ValueError("Set PRIMARY_RUN_ID and ABLATION_RUN_ID before evaluating OOF runs.")
if EVALUATION_RUN_ID is None:
    EVALUATION_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_operational_oof"
ABLATION_DIR = DRIVE_ROOT / "xgb_p_t_ablation_runs" / ABLATION_RUN_ID
RUN_DIRS = {
    "xgb_p": BASELINE_RUN_DIR,
    "current_window": ABLATION_DIR / "current_window",
    "history": ABLATION_DIR / "history",
    "full": DRIVE_ROOT / "xgb_p_t_runs" / PRIMARY_RUN_ID,
}
OUTPUT_DIR = DRIVE_ROOT / "operational_oof_runs" / EVALUATION_RUN_ID
print("Input runs:", RUN_DIRS)
print("Evaluation output:", OUTPUT_DIR)


Input runs: {'xgb_p': PosixPath('/content/drive/MyDrive/capture_gate0/xgb_p_runs/20260919T151844_852400Z_xgb_p'), 'current_window': PosixPath('/content/drive/MyDrive/capture_gate0/xgb_p_t_ablation_runs/20260920T000745_916321Z_xgb_p_t_ablation/current_window'), 'history': PosixPath('/content/drive/MyDrive/capture_gate0/xgb_p_t_ablation_runs/20260920T000745_916321Z_xgb_p_t_ablation/history'), 'full': PosixPath('/content/drive/MyDrive/capture_gate0/xgb_p_t_runs/20260919T201909_754628Z_xgb_p_t')}
Evaluation output: /content/drive/MyDrive/capture_gate0/operational_oof_runs/20260920T142514_611048Z_operational_oof


## 3. Compute or verify the immutable development report

The primary budget is one false-alert window per hour. Sensitivity budgets are one per 12 hours and one per five minutes. A window alerts when its maximum packet score is at least the model threshold. Only windows without malicious packets contribute false alerts and benign exposure; empty wall-clock windows contribute exposure but cannot alert. The threshold is the most permissive one whose worst fold mean scenario rate meets the budget. One attack-step iteration is detected only when one of its malicious packets crosses the threshold, at that packet window's close.


In [ ]:
if OUTPUT_DIR.exists():
    report = validate_operational_run(OUTPUT_DIR, MANIFEST_PATH, RUN_DIRS)
else:
    report = run_operational_oof_evaluation(
        manifest_path=MANIFEST_PATH, run_dirs=RUN_DIRS,
        output_dir=OUTPUT_DIR, batch_size=250_000)
print("Evaluation run ID:", EVALUATION_RUN_ID)
print("Budgets per hour:", report["budgets_per_hour"])


Aggregating xgb_p OOF windows and iterations...
Aggregating current_window OOF windows and iterations...
Aggregating history OOF windows and iterations...
Aggregating full OOF windows and iterations...
Evaluation run ID: 20260920T142514_611048Z_operational_oof
Budgets per hour: {'one_per_hour': 1.0, 'one_per_12_hours': 0.08333333333333333, 'one_per_5_minutes': 12.0}


## 4. Review thresholds and primary-budget metrics

Compare sequence detection rate and latency alongside false alerts. The detected-only latency is diagnostic; the main latency summary retains missed iterations using their attack-step duration. Review individual scenarios before interpreting the hierarchical macro result.


In [ ]:
budget_name = "one_per_hour"
threshold_rows = []
macro_rows = []
scenario_rows = []
for model_name, model_report in report["models"].items():
    selected = model_report["thresholds"][budget_name]
    threshold_rows.append({
        "model": model_name,
        "score_threshold": selected["threshold"],
        "worst_fold_false_alert_windows_per_hour": selected[
            "worst_fold_false_alert_windows_per_hour"],
    })
    result = model_report["budgets"][budget_name]
    macro_rows.append({"model": model_name, **result["hierarchical_macro"]})
    for scenario, metrics in result["scenario_metrics"].items():
        scenario_rows.append({"model": model_name, "scenario": scenario, **metrics})
display(pd.DataFrame(threshold_rows).set_index("model"))
display(pd.DataFrame(macro_rows).set_index("model"))
display(pd.DataFrame(scenario_rows).set_index(["model", "scenario"])[[
    "fold", "false_alert_windows_per_hour", "packet_recall",
    "sequence_detection_rate", "mean_miss_capped_latency_seconds",
    "attack_step_iterations", "detected_iterations"]])


,score_threshold,worst_fold_false_alert_windows_per_hour
model,,
xgb_p,0.999463,0.417543
current_window,0.983387,0.798887
history,0.995854,0.979633
full,0.997506,0.866198


,false_alert_windows_per_hour,packet_recall,packet_false_positive_rate,sequence_detection_rate,mean_miss_capped_latency_seconds
model,,,,,
xgb_p,0.208772,0.600363,0.000030,0.528501,63.809228
current_window,0.658179,0.827336,0.000128,0.909566,2.760956
history,0.918791,0.815902,0.000027,0.985104,3.081228
full,0.433099,0.771486,0.000068,0.844987,5.912042


fold  false_alert_windows_per_hour  \
model          scenario                                               
xgb_p          train_dollar_char    A                      0.495441   
               train_slash_char     A                      0.325792   
               train_sub_exf        A                      0.431396   
               train_empty_conn     B                      0.000000   
               train_qos_mid        B                      0.000000   
current_window train_dollar_char    A                      0.990883   
               train_slash_char     A                      0.542986   
               train_sub_exf        A                      0.862792   
               train_empty_conn     B                      0.707965   
               train_qos_mid        B                      0.326975   
history        train_dollar_char    A                      0.990883   
               train_slash_char     A                      0.977376   
               train_sub_exf        A                      0.970641   
               train_empty_conn     B                      1.061947   
               train_qos_mid        B                      0.653951   
full           train_dollar_char    A                      0.867022   
               train_slash_char     A                      0.868778   
               train_sub_exf        A                      0.862792   
               train_empty_conn     B                      0.000000   
               train_qos_mid        B                      0.000000   

                                  packet_recall  sequence_detection_rate  \
model          scenario                                                    
xgb_p          train_dollar_char       0.327993                 0.741379   
               train_slash_char        0.883428                 0.828652   
               train_sub_exf           0.536327                 0.570149   
               train_empty_conn        0.645770                 0.429907   
               train_qos_mid           0.590517                 0.257310   
current_window train_dollar_char       0.640818                 0.974138   
               train_slash_char        0.926638                 0.997191   
               train_sub_exf           0.940674                 0.710448   
               train_empty_conn        0.904870                 0.897196   
               train_qos_mid           0.732384                 0.953216   
history        train_dollar_char       0.613390                 1.000000   
               train_slash_char        0.949800                 0.997191   
               train_sub_exf           0.892591                 0.913433   
               train_empty_conn        0.876265                 1.000000   
               train_qos_mid           0.750156                 1.000000   
full           train_dollar_char       0.520783                 0.948276   
               train_slash_char        0.926562                 0.988764   
               train_sub_exf           0.920410                 0.614925   
               train_empty_conn        0.884988                 0.766355   
               train_qos_mid           0.622452                 0.912281   

                                  mean_miss_capped_latency_seconds  \
model          scenario                                              
xgb_p          train_dollar_char                         31.859447   
               train_slash_char                          22.649773   
               train_sub_exf                             33.582168   
               train_empty_conn                          71.641746   
               train_qos_mid                            124.867575   
current_window train_dollar_char                          3.258431   
               train_slash_char                           2.626940   
               train_sub_exf                              2.449528   
               train_empty_conn                           2.717260   
               train_qos_mi

## 5. Classical packet-classification metrics

  This section reports the packet-level metrics used in the earlier CIC2018 experiments: precision,
  recall, F1, F2, average precision, ROC-AUC, false-positive rate, and confusion-matrix counts.

  Metrics are evaluated at the existing operational thresholds. No additional threshold is selected.
  Results are reported per scenario, using the primary hierarchical macro aggregation, and as a pooled
  micro diagnostic.

  False-alert windows per hour are retained because packet false-positive rate alone can understate
  operational alert burden when millions of benign packets are evaluated.


In [ ]:
from utils.capture_data import load_manifest
from utils.capture_oof_operational import _validated_runs


PRIMARY_BUDGET = "one_per_hour"

manifest = load_manifest(MANIFEST_PATH)

fold_reports = _validated_runs(
    manifest,
    RUN_DIRS,
)


In [ ]:
import math


def classical_metrics_from_counts(
    true_positives,
    false_positives,
    true_negatives,
    false_negatives,
):
    """Compute CIC-style binary classification metrics from counts."""
    tp = int(true_positives)
    fp = int(false_positives)
    tn = int(true_negatives)
    fn = int(false_negatives)

    precision = (
        tp / (tp + fp)
        if tp + fp > 0
        else 0.0
    )
    recall = (
        tp / (tp + fn)
        if tp + fn > 0
        else 0.0
    )
    f1 = (
        2.0 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0.0
    )
    f2 = (
        5.0 * precision * recall
        / (4.0 * precision + recall)
        if precision + recall > 0
        else 0.0
    )
    false_positive_rate = (
        fp / (fp + tn)
        if fp + tn > 0
        else 0.0
    )
    false_negative_rate = (
        fn / (fn + tp)
        if fn + tp > 0
        else 0.0
    )
    specificity = (
        tn / (tn + fp)
        if tn + fp > 0
        else 0.0
    )
    accuracy = (
        (tp + tn) / (tp + fp + tn + fn)
        if tp + fp + tn + fn > 0
        else 0.0
    )
    balanced_accuracy = (
        recall + specificity
    ) / 2.0

    mcc_denominator = math.sqrt(
        (tp + fp)
        * (tp + fn)
        * (tn + fp)
        * (tn + fn)
    )
    matthews_correlation_coefficient = (
        (tp * tn - fp * fn) / mcc_denominator
        if mcc_denominator > 0
        else 0.0
    )

    return {
        "packet_precision": precision,
        "packet_recall": recall,
        "packet_f1": f1,
        "packet_f2": f2,
        "packet_false_positive_rate": false_positive_rate,
        "packet_false_negative_rate": false_negative_rate,
        "packet_specificity": specificity,
        "packet_accuracy": accuracy,
        "packet_balanced_accuracy": balanced_accuracy,
        "packet_mcc": matthews_correlation_coefficient,
        "packet_true_positives": tp,
        "packet_false_positives": fp,
        "packet_true_negatives": tn,
        "packet_false_negatives": fn,
    }


classical_scenario_rows = []

for model_name, model_report in report["models"].items():
    for budget_name in report["budget_order"]:
        budget_report = model_report["budgets"][budget_name]
        threshold = float(
            model_report["thresholds"][budget_name]["threshold"]
        )


        for fold, split in manifest["validation"]["folds"].items():
            for scenario in split["validate"]:
                scenario_metrics = budget_report[
                    "scenario_metrics"
                ][scenario]
                validation_metrics = fold_reports[
                    model_name
                ][fold]["validation"][scenario]

                count_metrics = classical_metrics_from_counts(
                    scenario_metrics["packet_true_positives"],
                    scenario_metrics["packet_false_positives"],
                    scenario_metrics["packet_true_negatives"],
                    scenario_metrics["packet_false_negatives"],
                )

                classical_scenario_rows.append({
                    "model": model_name,
                    "budget": budget_name,
                    "fold": fold,
                    "scenario": scenario,
                    "threshold": threshold,
                    "false_alert_windows_per_hour": (
                        scenario_metrics[
                            "false_alert_windows_per_hour"
                        ]
                    ),
                    "packet_roc_auc": float(
                        validation_metrics["packet_roc_auc"]
                    ),
                    "packet_average_precision": float(
                        validation_metrics[
                            "packet_pr_auc_diagnostic"
                        ]
                    ),
                    **count_metrics,
                })


classical_scenario_table = pd.DataFrame(
    classical_scenario_rows
)

classical_metric_columns = [
    "packet_precision",
    "packet_recall",
    "packet_f1",
    "packet_f2",
    "packet_roc_auc",
    "packet_average_precision",
    "packet_false_positive_rate",
    "packet_false_negative_rate",
    "packet_specificity",
    "packet_accuracy",
    "packet_balanced_accuracy",
    "packet_mcc",
    "false_alert_windows_per_hour",
]

classical_count_columns = [
    "packet_true_positives",
    "packet_false_positives",
    "packet_true_negatives",
    "packet_false_negatives",
]


classical_scenario_display_columns = [
    "model",
    "scenario",
    "fold",
    "budget",
    "threshold",
    "packet_precision",
    "packet_recall",
    "packet_f1",
    "packet_f2",
    "packet_roc_auc",
    "packet_average_precision",
    "packet_false_positive_rate",
    "false_alert_windows_per_hour",
    "packet_true_positives",
    "packet_false_positives",
    "packet_true_negatives",
    "packet_false_negatives",
]

display(
    classical_scenario_table[
        classical_scenario_table["budget"] == PRIMARY_BUDGET
    ][classical_scenario_display_columns]
    .sort_values(["model", "scenario"])
    .set_index(["model", "scenario"])
)



fold        budget  threshold  \
model          scenario                                          
current_window train_dollar_char    A  one_per_hour   0.983387   
               train_empty_conn     B  one_per_hour   0.983387   
               train_qos_mid        B  one_per_hour   0.983387   
               train_slash_char     A  one_per_hour   0.983387   
               train_sub_exf        A  one_per_hour   0.983387   
full           train_dollar_char    A  one_per_hour   0.997506   
               train_empty_conn     B  one_per_hour   0.997506   
               train_qos_mid        B  one_per_hour   0.997506   
               train_slash_char     A  one_per_hour   0.997506   
               train_sub_exf        A  one_per_hour   0.997506   
history        train_dollar_char    A  one_per_hour   0.995854   
               train_empty_conn     B  one_per_hour   0.995854   
               train_qos_mid        B  one_per_hour   0.995854   
               train_slash_char     A  one_per_hour   0.995854   
               train_sub_exf        A  one_per_hour   0.995854   
xgb_p          train_dollar_char    A  one_per_hour   0.999463   
               train_empty_conn     B  one_per_hour   0.999463   
               train_qos_mid        B  one_per_hour   0.999463   
               train_slash_char     A  one_per_hour   0.999463   
               train_sub_exf        A  one_per_hour   0.999463   

                                  packet_precision  packet_recall  packet_f1  \
model          scenario                                                        
current_window train_dollar_char          0.999475       0.640818   0.780936   
               train_empty_conn           0.999959       0.904870   0.950041   
               train_qos_mid              0.999964       0.732384   0.845509   
               train_slash_char           0.999876       0.926638   0.961865   
               train_sub_exf              0.999488       0.940674   0.969190   
full           train_dollar_char          0.999505       0.520783   0.684772   
               train_empty_conn           1.000000       0.884988   0.938985   
               train_qos_mid              1.000000       0.622452   0.767298   
               train_slash_char           0.999924       0.926562   0.961847   
               train_sub_exf              0.999873       0.920410   0.958497   
history        train_dollar_char          0.999939       0.613390   0.760357   
               train_empty_conn           0.999949       0.876265   0.934030   
               train_qos_mid              0.999958       0.750156   0.857229   
               train_slash_char           0.999989       0.949800   0.974248   
               train_sub_exf              0.999923       0.892591   0.943213   
xgb_p          train_dollar_char          0.999987       0.327993   0.493967   
               train_empty_conn           0.999845       0.645770   0.784716   
               train_qos_mid              0.999903       0.590517   0.742521   
               train_slash_char           0.999999       0.883428   0.938106   
               train_sub_exf              0.999982       0.536327   0.698189   

                                  packet_f2  packet_roc_auc  \
model          scenario                                       
current_window train_dollar_char   0.690365        0.997493   
               train_empty_conn    0.922413        0.999148   
               train_qos_mid       0.773796        0.998877   
               train_slash_char    0.940415        0.998167   
               train_sub_exf       0.951877        0.999038   
full           train_dollar_char   0.575955        0.994512   
               train_empty_conn    0.905824        0.999476   
               train_qos_mid       0.673292        0.997640   
               train_slash_char    0.940361        0.996789   
               train_sub_exf       0.935275        0.998353   
history        train_dollar_char   0.664788        0.995527   


In [ ]:
classical_fold_table = (
    classical_scenario_table
    .groupby(
        ["model", "budget", "fold"],
        as_index=False,
        sort=False,
    )[classical_metric_columns]
    .mean()
)

classical_macro_table = (
    classical_fold_table
    .groupby(
        ["model", "budget"],
        as_index=False,
        sort=False,
    )[classical_metric_columns]
    .mean()
)

threshold_table = (
    classical_scenario_table
    .groupby(
        ["model", "budget"],
        as_index=False,
        sort=False,
    )["threshold"]
    .first()
)

classical_macro_table = (
    classical_macro_table
    .merge(
        threshold_table,
        on=["model", "budget"],
        how="left",
        validate="one_to_one",
    )
)

classical_macro_display_columns = [
    "model",
    "budget",
    "threshold",
    "packet_precision",
    "packet_recall",
    "packet_f1",
    "packet_f2",
    "packet_roc_auc",
    "packet_average_precision",
    "packet_false_positive_rate",
    "false_alert_windows_per_hour",
    "packet_balanced_accuracy",
    "packet_mcc",
]

display(
    classical_macro_table[
        classical_macro_display_columns
    ]
    .sort_values(["budget", "model"])
    .set_index(["budget", "model"])
)


threshold  packet_precision  packet_recall  \
budget            model                                                        
one_per_12_hours  current_window   0.994354          0.999969       0.794438   
                  full             0.998009          0.999977       0.766824   
                  history          0.999206          0.999998       0.721846   
                  xgb_p            0.999942          0.999999       0.518892   
one_per_5_minutes current_window   0.959686          0.999027       0.846149   
                  full             0.996420          0.999637       0.780297   
                  history          0.988180          0.999538       0.869324   
                  xgb_p            0.987051          0.999658       0.733198   
one_per_hour      current_window   0.983387          0.999787       0.827336   
                  full             0.997506          0.999884       0.771486   
                  history          0.995854          0.999952       0.815902   
                  xgb_p            0.999463          0.999932       0.600363   

                                  packet_f1  packet_f2  packet_roc_auc  \
budget            model                                                  
one_per_12_hours  current_window   0.879069   0.825248        0.998623   
                  full             0.857150   0.799012        0.997555   
                  history          0.830822   0.760681        0.998556   
                  xgb_p            0.672392   0.570186        0.969854   
one_per_5_minutes current_window   0.912280   0.870728        0.998623   
                  full             0.867213   0.811477        0.997555   
                  history          0.928259   0.891663        0.998556   
                  xgb_p            0.841427   0.772261        0.969854   
one_per_hour      current_window   0.900886   0.854495        0.998623   
                  full             0.860757   0.803378        0.997555   
                  history          0.894118   0.844757        0.998556   
                  xgb_p            0.736853   0.646975        0.969854   

                                  packet_average_precision  \
budget            model                                      
one_per_12_hours  current_window                  0.998205   
                  full                            0.996965   
                  history                         0.997909   
                  xgb_p                           0.968574   
one_per_5_minutes current_window                  0.998205   
                  full                            0.996965   
                  history                         0.997909   
                  xgb_p                           0.968574   
one_per_hour      current_window                  0.998205   
                  full                            0.996965   
                  history                         0.997909   
                  xgb_p                           0.968574   

                                  packet_false_positive_rate  \
budget            model                                        
one_per_12_hours  current_window                1.991183e-05   
                  full                          1.104592e-05   
                  history                       1.158276e-06   
                  xgb_p                         2.932546e-07   
one_per_5_minutes current_window                5.335559e-04   
                  full                          1.826829e-04   
                  history                       2.594214e-04   
                  xgb_p                         1.547785e-04   
one_per_hour      current_window                1.282779e-04   
                  full                          6.803507e-05   
                  history                       2.720009e-05   
                  xgb_p                         3.025554e-05   

                                  false_alert_windows_per_hour  \
budget            model                     

In [ ]:
classical_micro_count_table = (
    classical_scenario_table
    .groupby(
        ["model", "budget"],
        as_index=False,
        sort=False,
    )[classical_count_columns]
    .sum()
)

classical_micro_rows = []

for row in classical_micro_count_table.to_dict(
    orient="records"
):
    model_name = row["model"]
    budget_name = row["budget"]

    count_metrics = classical_metrics_from_counts(
        row["packet_true_positives"],
        row["packet_false_positives"],
        row["packet_true_negatives"],
        row["packet_false_negatives"],
    )

    matching_macro = classical_macro_table[
        (classical_macro_table["model"] == model_name)
        & (
            classical_macro_table["budget"]
            == budget_name
        )
    ].iloc[0]

    classical_micro_rows.append({
        "model": model_name,
        "budget": budget_name,
        "threshold": float(
            matching_macro["threshold"]
        ),
        "false_alert_windows_per_hour": float(
            matching_macro[
                "false_alert_windows_per_hour"
            ]
        ),
        **count_metrics,
    })


classical_micro_table = pd.DataFrame(
    classical_micro_rows
)

classical_micro_display_columns = [
    "model",
    "budget",
    "threshold",
    "packet_precision",
    "packet_recall",
    "packet_f1",
    "packet_f2",
    "packet_false_positive_rate",
    "false_alert_windows_per_hour",
    "packet_true_positives",
    "packet_false_positives",
    "packet_true_negatives",
    "packet_false_negatives",
]

display(
    classical_micro_table[
        classical_micro_display_columns
    ]
    .sort_values(["budget", "model"])
    .set_index(["budget", "model"])
)



threshold  packet_precision  packet_recall  \
budget            model                                                        
one_per_12_hours  current_window   0.994354          0.999970       0.843396   
                  full             0.998009          0.999982       0.826674   
                  history          0.999206          0.999999       0.811951   
                  xgb_p            0.999942          0.999999       0.593513   
one_per_5_minutes current_window   0.959686          0.999257       0.873663   
                  full             0.996420          0.999707       0.837674   
                  history          0.988180          0.999714       0.909170   
                  xgb_p            0.987051          0.999807       0.822912   
one_per_hour      current_window   0.983387          0.999813       0.862848   
                  full             0.997506          0.999890       0.830804   
                  history          0.995854          0.999974       0.869835   
                  xgb_p            0.999463          0.999982       0.731166   

                                  packet_f1  packet_f2  \
budget            model                                  
one_per_12_hours  current_window   0.915034   0.870662   
                  full             0.905107   0.856357   
                  history          0.896217   0.843681   
                  xgb_p            0.744912   0.646034   
one_per_5_minutes current_window   0.932249   0.896191   
                  full             0.911546   0.865738   
                  history          0.952295   0.925942   
                  xgb_p            0.902775   0.853099   
one_per_hour      current_window   0.926295   0.887155   
                  full             0.907539   0.859886   
                  history          0.930376   0.893080   
                  xgb_p            0.844703   0.772710   

                                  packet_false_positive_rate  \
budget            model                                        
one_per_12_hours  current_window                2.935559e-05   
                  full                          1.709888e-05   
                  history                       1.059222e-06   
                  xgb_p                         4.539525e-07   
one_per_5_minutes current_window                7.470545e-04   
                  full                          2.820558e-04   
                  history                       2.988521e-04   
                  xgb_p                         1.830942e-04   
one_per_hour      current_window                1.853639e-04   
                  full                          1.053170e-04   
                  history                       2.632924e-05   
                  xgb_p                         1.528307e-05   

                                  false_alert_windows_per_hour  \
budget            model                                          
one_per_12_hours  current_window                      0.000000   
                  full                                0.036074   
                  history                             0.036074   
                  xgb_p                               0.000000   
one_per_5_minutes current_window                      5.655613   
                  full                                4.158372   
                  history                             6.718528   
                  xgb_p                               3.163539   
one_per_hour      current_window                      0.658179   
                  full                                0.433099   
                  history                             0.918791   
                  xgb_p                               0.208772   

                                  packet_true_positives  \
budget            model                                   
one_per_12_hours  current_window                6410448   
                  full                          6283346   
                  history                 

In [ ]:
cic_style_primary_table = (
    classical_macro_table[
        classical_macro_table["budget"]
        == PRIMARY_BUDGET
    ][[
        "model",
        "threshold",
        "packet_precision",
        "packet_recall",
        "packet_f1",
        "packet_f2",
        "packet_average_precision",
        "packet_roc_auc",
        "packet_false_positive_rate",
        "false_alert_windows_per_hour",
    ]]
    .rename(columns={
        "packet_precision": "Precision",
        "packet_recall": "Recall",
        "packet_f1": "F1",
        "packet_f2": "F2",
        "packet_average_precision": "AUC-PR",
        "packet_roc_auc": "AUC-ROC",
        "packet_false_positive_rate": "FPR",
    })
    .set_index("model")
)

display(cic_style_primary_table)


,threshold,Precision,Recall,F1,F2,AUC-PR,AUC-ROC,FPR,false_alert_windows_per_hour
model,,,,,,,,,
xgb_p,0.999463,0.999932,0.600363,0.736853,0.646975,0.968574,0.969854,0.000030,0.208772
current_window,0.983387,0.999787,0.827336,0.900886,0.854495,0.998205,0.998623,0.000128,0.658179
history,0.995854,0.999952,0.815902,0.894118,0.844757,0.997909,0.998556,0.000027,0.918791
full,0.997506,0.999884,0.771486,0.860757,0.803378,0.996965,0.997555,0.000068,0.433099


## 6. Review budget sensitivity and weak attack steps

The two sensitivity budgets use thresholds selected by the same rule. Missed iterations remain in the saved report for independent review.


In [ ]:
sensitivity_rows = []
for model_name, model_report in report["models"].items():
    for budget_name in report["budget_order"]:
        selected = model_report["thresholds"][budget_name]
        summary = model_report["budgets"][budget_name]["hierarchical_macro"]
        sensitivity_rows.append({"model": model_name, "budget": budget_name,
                                 "threshold": selected["threshold"], **summary})
display(pd.DataFrame(sensitivity_rows).set_index(["model", "budget"]))
step_rows = []
for model_name, model_report in report["models"].items():
    for step in model_report["budgets"]["one_per_hour"]["step_metrics"].values():
        step_rows.append({"model": model_name, **step})
display(pd.DataFrame(step_rows).sort_values(
    ["sequence_detection_rate", "mean_miss_capped_latency_seconds"],
    ascending=[True, False]).head(40))


threshold  false_alert_windows_per_hour  \
model          budget                                                       
xgb_p          one_per_hour        0.999463                      0.208772   
               one_per_12_hours    0.999942                      0.000000   
               one_per_5_minutes   0.987051                      3.163539   
current_window one_per_hour        0.983387                      0.658179   
               one_per_12_hours    0.994354                      0.000000   
               one_per_5_minutes   0.959686                      5.655613   
history        one_per_hour        0.995854                      0.918791   
               one_per_12_hours    0.999206                      0.036074   
               one_per_5_minutes   0.988180                      6.718528   
full           one_per_hour        0.997506                      0.433099   
               one_per_12_hours    0.998009                      0.036074   
               one_per_5_minutes   0.996420                      4.158372   

                                  packet_recall  packet_false_positive_rate  \
model          budget                                                         
xgb_p          one_per_hour            0.600363                3.025554e-05   
               one_per_12_hours        0.518892                2.932546e-07   
               one_per_5_minutes       0.733198                1.547785e-04   
current_window one_per_hour            0.827336                1.282779e-04   
               one_per_12_hours        0.794438                1.991183e-05   
               one_per_5_minutes       0.846149                5.335559e-04   
history        one_per_hour            0.815902                2.720009e-05   
               one_per_12_hours        0.721846                1.158276e-06   
               one_per_5_minutes       0.869324                2.594214e-04   
full           one_per_hour            0.771486                6.803507e-05   
               one_per_12_hours        0.766824                1.104592e-05   
               one_per_5_minutes       0.780297                1.826829e-04   

                                  sequence_detection_rate  \
model          budget                                       
xgb_p          one_per_hour                      0.528501   
               one_per_12_hours                  0.395452   
               one_per_5_minutes                 0.994527   
current_window one_per_hour                      0.909566   
               one_per_12_hours                  0.838935   
               one_per_5_minutes                 0.974778   
history        one_per_hour                      0.985104   
               one_per_12_hours                  0.860758   
               one_per_5_minutes                 0.995522   
full           one_per_hour                      0.844987   
               one_per_12_hours                  0.827461   
               one_per_5_minutes                 0.863420   

                                  mean_miss_capped_latency_seconds  
model          budget                                               
xgb_p          one_per_hour                              63.809228  
               one_per_12_hours                          70.745783  
               one_per_5_minutes                          2.584426  
current_window one_per_hour                               2.760956  
               one_per_12_hours                           4.838608  
               one_per_5_minutes                          2.678593  
history        one_per_hour                               3.081228  
               one_per_12_hours                           9.643856  
               one_per_5_minutes                          2.563735  
full           one_per_hour                               5.912042  
               one_per_12_hours                           6.515564  
               one_per_5_minutes                          4.334954

,model,scenario,attack_step,iterations,detected_iterations,sequence_detection_rate,mean_miss_capped_latency_seconds
25,xgb_p,train_empty_conn,empty_conn,37,0,0.000000,94.925403
39,xgb_p,train_qos_mid,qos_mid,39,0,0.000000,20.590014
18,xgb_p,train_sub_exf,mqtt_cat,27,0,0.000000,5.365242
27,xgb_p,train_empty_conn,mqtt_cat,12,0,0.000000,5.302040
34,xgb_p,train_qos_mid,mqtt_cat,10,0,0.000000,5.222620
41,xgb_p,train_qos_mid,scp_inst,5,0,0.000000,0.199340
133,full,train_dollar_char,scp_inst,7,0,0.000000,0.192010
23,xgb_p,train_sub_exf,scp_exf,117,0,0.000000,0.184702
15,xgb_p,train_slash_char,sftp_inst,1,0,0.000000,0.166788
141,full,train_slash_char,sftp_inst,1,0,0.000000,0.166788
